# 1. 数据集处理

### 1.1 SCDHD数据集

In [ ]:
import wfdb
import numpy as np
from scipy.signal import resample
from torch.utils.data import DataLoader, TensorDataset
import torch
import os

# 全局配置
fs = 250  # 原始采样频率(Hz)
target_fs = 128  # 目标采样频率(Hz)
window_size = 30 * 60 * fs  # 30分钟窗口大小（以样本数表示）
step_size = 1 * 60 * fs  # 1分钟滑动步长（以样本数表示）
dataset_path = os.environ.get("SCD_PATH", os.path.join("data", "SCD"))  # 数据集绝对路径

# 根据表格创建时间配置字典
record_config = {
    '30': {'type': 'SCD',       'start': '6:54:33',     'end': '7:54:33'},
    '31': {'type': 'SCD',       'start': '12:42:24',    'end': '13:42:24'},
    '32': {'type': 'SCD',       'start': '15:45:18',    'end': '16:45:18'},
    '33': {'type': 'SCD',       'start': '3:46:19',     'end': '4:46:19'},
    '34': {'type': 'SCD',       'start': '5:35:44',     'end': '6:35:44'},
    '35': {'type': 'SCD',       'start': '23:34:56',    'end': '24:34:56'},
    '36': {'type': 'SCD',       'start': '17:59:01',    'end': '18:59:01'},
    '37': {'type': 'SCD',       'start': '0:31:13',     'end': '1:31:13'},
    '38': {'type': 'SCD',       'start': '7:01:54',     'end': '8:01:54'},
    '39': {'type': 'SCD',       'start': '3:37:51',     'end': '4:37:51'},
    '40': {'type': 'noramal',   'start': 1*3600,        'end': 2*3600},  # 特殊处理1-2小时
    '41': {'type': 'SCD',       'start': '1:59:24',     'end': '2:59:24'},
    '42': {'type': 'normal',    'start': 1*3600,        'end': 2*3600},
    '43': {'type': 'SCD',       'start': '14:37:11',    'end': '15:37:11'},
    '44': {'type': 'SCD',       'start': '18:38:45',    'end': '19:38:45'},
    '45': {'type': 'SCD',       'start': '17:09:17',    'end': '18:09:17'},
    '46': {'type': 'SCD',       'start': '2:41:47',     'end': '3:41:47'},
    '47': {'type': 'SCD',       'start': '5:13:01',     'end': '6:13:01'},
    '48': {'type': 'SCD',       'start': '1:29:40',     'end': '2:29:40'},
    '49': {'type': 'normal',    'start': 1*3600,        'end': 2*3600},
    '50': {'type': 'SCD',       'start': '10:45:43',    'end': '11:45:43'},
    '51': {'type': 'SCD',       'start': '21:58:23',    'end': '22:58:23'},
    '52': {'type': 'SCD',       'start': '1:32:40',     'end': '2:32:40'}
}

In [ ]:
def time_to_seconds(timestr):
    """将时:分:秒或纯秒数转换为总秒数"""
    if isinstance(timestr, int) or isinstance(timestr, float):
        return int(timestr)
    try:
        parts = list(map(int, timestr.split(':')))
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    except:
        raise ValueError(f"无效时间格式: {timestr}")

def generate_labels(signals):
    """生成标签向量"""
    labels = []  # 初始化一个空列表，用于存储生成的标签
    for i in range(0, len(signals) - window_size, step_size):  # 遍历信号，步长为step_size
        n = (i // step_size) + 1  # 计算当前窗口的序号
        n = min(n, 30)  # 确保n不超过30
        # 生成标签：30 - n个0，n个1
        label = [0] * (30 - n) + [1] * n  # 生成标签向量
        labels.append(label)  # 将生成的标签添加到列表中
    return np.array(labels, dtype=np.float32)  # 将标签列表转换为NumPy数组，并指定数据类型为float32

def resample_window(window, original_fs, target_fs):
    """重采样窗口数据"""
    num_samples = int(len(window) * target_fs / original_fs)
    return resample(window, num_samples)

In [ ]:
def check_nan_info(name, array):
    total = np.prod(array.shape)
    nan_count = np.isnan(array).sum()
    inf_count = np.isinf(array).sum()
    zero_count = np.sum(array == 0)

    if nan_count > 0:
        print(f"[N警告] {name} 中存在 NaN: {nan_count}/{total} ({nan_count/total:.4%})")
    if inf_count > 0:
        print(f"[I警告] {name} 中存在 Inf: {inf_count}/{total} ({inf_count/total:.4%})")
    if zero_count == total:
        print(f"[值警告] {name} 全部为0")
    elif zero_count > 0:
        print(f"[值提醒] {name} 中包含 0: {zero_count}/{total} ({zero_count/total:.4%})")



In [ ]:
def fix_nan_with_median(signals):
    """对信号中 NaN/Inf 进行中值填充修复"""
    fixed = signals.copy()
    for ch in range(fixed.shape[1]):
        channel_data = fixed[:, ch]
        finite_mask = np.isfinite(channel_data)
        if not finite_mask.all():
            if np.any(finite_mask):
                median_val = np.median(channel_data[finite_mask])
            else:
                median_val = 0.0  # 如果整列都是NaN/Inf，用0替代
                print(f"⚠️警告：通道{ch} 全为NaN/Inf，使用0填充")
            # 替换 NaN 和 Inf
            channel_data = np.where(np.isfinite(channel_data), channel_data, median_val)
            fixed[:, ch] = channel_data
    return fixed

def fix_nan_1d_with_median(signal_1d):
    """对1D信号中的 NaN 和 Inf 进行中值填充"""
    finite_mask = np.isfinite(signal_1d)
    if not finite_mask.all():
        if np.any(finite_mask):
            median_val = np.median(signal_1d[finite_mask])
        else:
            median_val = 0.0
            print("⚠️ 警告：该信号全为 NaN/Inf，用 0 填充")
        # 替换 NaN 和 Inf
        signal_1d = np.where(np.isfinite(signal_1d), signal_1d, median_val)
    return signal_1d


### 1.2 去噪

In [ ]:
from scipy.signal import butter, filtfilt

# 定义低通滤波函数（4阶Butterworth，截止频率40Hz）
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs  # 奈奎斯特频率
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

# 定义高通滤波函数（4阶Butterworth，截止频率0.5Hz）
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, data)
    return y

In [ ]:
def process_record(record_id):
    """处理单个记录"""
    config = record_config.get(record_id)
    
    if not config:
        print(f"跳过未配置的记录: {record_id}")
        return None, None

    try:
        # 检查文件是否存在
        record_path = os.path.join(dataset_path, record_id)
        if not os.path.exists(f"{record_path}.hea") or not os.path.exists(f"{record_path}.dat"):
            print(f"错误: 记录{record_id}的数据文件不存在")
            return None, None

        # record 40,42,49 特殊处理
        if config['type'] == 'normal':
            start_sec = config['start']
            end_sec = config['end']
            event_time = None  # 正常记录无事件时间
        else:
            start_sec = time_to_seconds(config['start'])
            end_sec = time_to_seconds(config['end'])
            event_time = time_to_seconds(config['end'])  # 事件时间为截取结束时间

        start_sample = int(start_sec * fs)
        end_sample = int(end_sec * fs)

        # 读取原始数据，使用Lead II（通道索引为1）
        signals, fields = wfdb.rdsamp(record_path, channels=[1])

        # 检查NaN------------------------原始信号读取后-------------------------
        # check_nan_info(f"{record_id} 原始signals", signals)
        
        # 验证数据长度
        total_samples = fields['sig_len']
        if end_sample > total_samples:
            print(f"警告: 记录{record_id}总时长不足，将截取到文件末尾")
            end_sample = total_samples

        # 执行截取
        signals_cut = signals[start_sample:end_sample]

        # 检查NaN------------------------截取后的信号-------------------------
        # check_nan_info(f"{record_id} signals_cut", signals_cut)

        
        # 中值填补NaN/Inf
        signals_cut = fix_nan_with_median(signals_cut)

        # 3. 检查修复后是否仍有异常
        check_nan_info(f"{record_id} signals_cut(中值填充后)", signals_cut)

         # ===== 新增去噪处理 =====
        # 检查信号长度是否满足滤波要求（padlen通常约为15）
        if signals_cut.shape[0] >= 15:
            # 假设signals_cut为二维数组 (n_samples, n_channels)
            # 对每个通道分别进行滤波处理
            for ch in range(signals_cut.shape[1]):
                # 先低通，再高通
                filtered = butter_lowpass_filter(signals_cut[:, ch], cutoff=40, fs=fs, order=4)
                filtered = butter_highpass_filter(filtered, cutoff=0.5, fs=fs, order=4)
                signals_cut[:, ch] = filtered
        else:
            print(f"警告：记录{record_id}截取的信号长度不足，未执行滤波去噪处理")
        # ==========================
        
        # 生成标签
        labels = generate_labels(signals_cut)

        # 检查NaN------------------------滤波后-------------------------
        check_nan_info(f"{record_id} 滤波后", signals_cut)
        
        # 生成窗口数据并重采样
        X = []
        for i in range(0, len(signals_cut) - window_size, step_size):
            window = signals_cut[i:i + window_size]
            check_nan_info(f"{record_id} 第{i//step_size}个窗口(滤波后)", window)

            window_resampled = resample_window(window, fs, target_fs)
            check_nan_info(f"{record_id} 第{i//step_size}个窗口(重采样后)", window_resampled)
            
            X.append(window_resampled)
        X = np.array(X)
        
        return X, labels

    except Exception as e:
        print(f"处理记录{record_id}时发生错误: {str(e)}")
        return None, None

# 处理所有记录并生成DataLoader
all_X = []
all_Y = []
for record_id in record_config.keys():
    X, Y = process_record(record_id)
    if X is not None and Y is not None:
        all_X.append(X)
        all_Y.append(Y)

# 检查是否加载了数据
if len(all_X) == 0 or len(all_Y) == 0:
    print("错误: 未加载任何数据，请检查文件路径和配置")
else:
    # 检查维度是否一致
    if not all(y.ndim == all_Y[0].ndim for y in all_Y):
        print("错误: all_Y中的数组维度不一致，无法合并")
    else:
        # 合并所有记录的数据
        all_X = np.concatenate(all_X, axis=0)
        all_Y = np.concatenate(all_Y, axis=0)

        # 转换为PyTorch张量
        X_tensor = torch.tensor(all_X, dtype=torch.float32)
        Y_tensor = torch.tensor(all_Y, dtype=torch.float32)

        # 创建DataLoader
        dataset = TensorDataset(X_tensor, Y_tensor)
        dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

        print(f"DataLoader创建完成，共{len(dataset)}个样本")

In [ ]:
# 检查原始信号中是否有 NaN / Inf
nan_count = torch.isnan(X_tensor).sum().item()
inf_count = torch.isinf(X_tensor).sum().item()

print(f"[原始数据] 含 NaN 样本数: {nan_count}, 含 Inf 样本数: {inf_count}")

In [ ]:
print(X_tensor.shape)
print(Y_tensor.shape)

In [ ]:
# 检查每条记录的样本数
for i, (X, Y) in enumerate(zip(all_X, all_Y)):
    print(f"Num：{i+1}: 输入样本数={len(X)}, 标签样本数={len(Y)}")

# 检查标签内容
print("第一条记录的标签:")
print(all_Y[0])

# 检查数据形状
print("所有输入的形状是否一致:", all(x.shape == all_X[0].shape for x in all_X))
print("所有标签的形状是否一致:", all(y.shape == all_Y[0].shape for y in all_Y))

### 1.2 MIT-BIH数据集

In [ ]:
import os
import numpy as np
import wfdb

# 数据集路径
data_path = os.environ.get("NSR_PATH", os.path.join("data", "NSR"))

# 参数设置
fs = 128  # 采样频率
start_hour = 1  # 起始小时
end_hour = 2  # 结束小时
window_size = 30 * 60 * fs  # 窗口大小（30分钟）
step_size = 1 * 60 * fs  # 滑动步长（1分钟）
num_windows = 30  # 每条记录的窗口数

# 初始化存储
M_X = []  # 存储所有窗口数据
M_y = []  # 存储所有标签

# 遍历所有记录
for record_name in os.listdir(data_path):
    if record_name.endswith(".dat"):
        record_path = os.path.join(data_path, record_name[:-4])  # 去掉.dat后缀
        print(f"正在处理记录: {record_name}")

        # 读取记录
        record = wfdb.rdrecord(record_path)
        signals = record.p_signal[:, 1]  # 获取lead 2 通道的信号

        # 检查记录长度是否足够
        required_length = end_hour * 60 * 60 * fs
        if len(signals) < required_length:
            print(f"记录 {record_name} 长度不足，跳过处理")
            continue

        # 截取1小时到2小时之间的数据
        start_idx = start_hour * 60 * 60 * fs
        end_idx = end_hour * 60 * 60 * fs
        signals_cut = signals[start_idx:end_idx]

        signals_cut = fix_nan_1d_with_median(signals_cut)
        
        # 可选：检查修复后的信号是否干净
        check_nan_info(f"{record_name} signals_cut(中值修复后)", signals_cut)
        
        print(f"截取信号长度: {len(signals_cut)}")

         # ===== 新增去噪处理 =====
        # 检查信号长度是否满足滤波要求（padlen通常约为15）
        if signals_cut.shape[0] >= 15:
            # 假设signals_cut为二维数组 (n_samples, n_channels)
            # 对每个通道分别进行滤波处理
            if signals_cut.ndim == 1:
                filtered = butter_lowpass_filter(signals_cut, cutoff=40, fs=fs, order=4)
                filtered = butter_highpass_filter(filtered, cutoff=0.5, fs=fs, order=4)
                signals_cut = filtered  # 重写为去噪后的信号
        else:
            print(f"警告：记录{record_name}截取的信号长度不足，未执行滤波去噪处理")
        # ==========================

        # 滑动窗口处理
        for i in range(0, len(signals_cut) - window_size + 1, step_size):
            window = signals_cut[i:i + window_size]
            M_X.append(window)
            M_y.append([0]*30)  # 标签为0

# 转换为NumPy数组
M_X_np = np.array(M_X)
M_y_np = np.array(M_y)

# 输出结果
print(f"总窗口数: {len(M_X)}")
print(f"每个窗口的样本数: {len(M_X[0])}")
print(f"标签总数: {len(M_y)}")

### 1.3 数据集合并

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 假设SCD数据集和MITBIH数据集已经准备好
# M_X_scd, M_y_scd: SCD数据集的数据和标签
# M_X_mitbih, M_y_mitbih: MITBIH数据集的数据和标签

# 1. 转换为PyTorch张量
X_scd_tensor = X_tensor  # SCD数据转换为张量
y_scd_tensor = Y_tensor  # SCD标签转换为张量
X_mitbih_tensor = torch.tensor(M_X_np , dtype=torch.float32)  # MITBIH数据转换为张量
y_mitbih_tensor = torch.tensor(M_y_np , dtype=torch.float32)  # MITBIH标签转换为张量

In [ ]:
def check_nan(name, tensor):
    if torch.isnan(tensor).any():
        print(f"[!] NaN detected in {name}")
    if torch.isinf(tensor).any():
        print(f"[!] Inf detected in {name}")

check_nan("X_scd_tensor", X_scd_tensor)
check_nan("y_scd_tensor", y_scd_tensor)
check_nan("X_mitbih_tensor", X_mitbih_tensor)
check_nan("y_mitbih_tensor", y_mitbih_tensor)

In [ ]:
print(X_scd_tensor.shape)
print(y_scd_tensor.shape)

In [ ]:
print(X_mitbih_tensor.shape)
print(y_mitbih_tensor.shape)

In [ ]:
X_mitbih_tensor = X_mitbih_tensor.unsqueeze(2)  # 在第2维度上增加一个维度

# 2. 合并数据集
X_combined = torch.cat([X_scd_tensor, X_mitbih_tensor], dim=0)  # 合并数据
y_combined = torch.cat([y_scd_tensor, y_mitbih_tensor], dim=0)  # 合并标签


In [ ]:
# 3. 创建自定义数据集
dataset = TensorDataset(X_combined, y_combined)

# 4. 使用DataLoader加载数据
batch_size = 32  # 批量大小
shuffle = True   # 是否随机打乱数据
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

# 打印信息
print(f"合并后的数据集大小: {len(dataset)}")
print(f"DataLoader的批量大小: {batch_size}")


### 1.4 数据集保存

In [ ]:
# 保存
torch.save(dataset, 'early warning dataset.pth')
config = {'batch_size': batch_size, 'shuffle': shuffle}
torch.save(config, 'early warning dataloader_config.pth')


### 1.5 数据集加载

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.font_manager import FontProperties
font_path = os.environ.get('FONT_PATH')
if font_path and os.path.exists(font_path):
    font_prop = FontProperties(fname=font_path)
    plt.rcParams['font.sans-serif'] = [font_prop.get_name()]
else:
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
import torch
from torch.utils.data import TensorDataset, DataLoader
# 加载
dataset = torch.load('early warning dataset.pth')
config = torch.load('early warning dataloader_config.pth')
dataloader = DataLoader(dataset, **config)

# 验证
print(f"数据集大小: {len(dataset)}")
print(f"DataLoader 配置: {config}")
# 从 DataLoader 中取出一个批次
for batch in dataloader:
    inputs, labels = batch  # 假设 TensorDataset 的构造为 (inputs, labels)
    break


In [ ]:
# 测试数据集是否正确
# 如果 inputs 为二维：batch_size x signal_length
# 如果 inputs 为三维：batch_size x signal_length x channels，则取第一个通道
signal = inputs[0].detach().cpu().numpy()  # 选取第一个样本

# 如果数据是二维 (n,) ，直接绘图，如果是二维以上则需要选择通道
if signal.ndim > 1:
    # 这里选择第一个通道
    signal = signal[:, 0]
print(len(signal))
plt.figure(figsize=(24, 4))
plt.plot(signal[:230400])
plt.title("示例心电信号")
plt.xlabel("时间 (采样点)")
plt.ylabel("幅值")
plt.show()

In [ ]:
print(f"输入批次形状: {inputs.shape}")

### 1.6 裁剪 ＋ CWT

In [ ]:
import torch
import numpy as np
import pywt
from torch.utils.data import DataLoader
from tqdm import tqdm

def apply_cwt(signal, num_scales=36, wavelet='morl', downsample_factor=8):
    """
    signal: 1D torch tensor, shape (T,)
    returns: CWT tensor of shape (num_scales, T // downsample_factor)
    """
    import pywt
    import torch.nn.functional as F

    signal_np = signal.cpu().numpy()
    scales = np.arange(1, num_scales + 1)
    cwt_coeffs, _ = pywt.cwt(signal_np, scales, wavelet)
    cwt_coeffs = np.abs(cwt_coeffs)  # (S, T)

    cwt_tensor = torch.tensor(cwt_coeffs, dtype=torch.float32)  # (S, T)

    # 平均池化降采样（推荐）
    x = cwt_tensor.unsqueeze(0).unsqueeze(0)  # (1, 1, S, T)
    x_pooled = F.avg_pool2d(x, kernel_size=(1, downsample_factor), stride=(1, downsample_factor))
    cwt_downsampled = x_pooled.squeeze(0).squeeze(0)  # (S, T//ds)
    return cwt_downsampled


def process_batch_with_cwt(batch_inputs, fs=128, win_sec=60, num_scales=36):
    """
    输入：
        batch_inputs: torch tensor, shape (B, 230400)
    输出：
        all_cwt: shape (B, 30, num_scales, T')，T' ≈ 7680
    """
    B, total_len = batch_inputs.shape
    win_len = fs * win_sec  # 每分钟点数
    num_windows = total_len // win_len
    result = []

    for b in tqdm(range(B), desc="CWT per record"):
        sample = batch_inputs[b]  # shape (230400,)
        windows = []
        for i in range(num_windows):
            segment = sample[i * win_len : (i + 1) * win_len]  # shape (7680,)
            cwt_tensor = apply_cwt(segment, num_scales=num_scales)  # (num_scales, 7680)
            windows.append(cwt_tensor)
        windows = torch.stack(windows)  # shape (30, num_scales, 7680)
        result.append(windows)

    all_cwt = torch.stack(result)  # shape (B, 30, num_scales, 7680)
    return all_cwt

# 单条测试

In [ ]:
all_cwt = process_batch_with_cwt(inputs.squeeze(-1), fs=128, win_sec=60, num_scales=36)
print("CWT 输出 shape:", all_cwt.shape)

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# 你的 CWT 函数（含下采样）
def apply_cwt(signal, num_scales=36, wavelet='morl', downsample_factor=8):
    import pywt
    import torch.nn.functional as F

    signal_np = signal.cpu().numpy()
    scales = np.arange(1, num_scales + 1)
    cwt_coeffs, _ = pywt.cwt(signal_np, scales, wavelet)
    cwt_coeffs = np.abs(cwt_coeffs)
    cwt_tensor = torch.tensor(cwt_coeffs, dtype=torch.float32)

    # 下采样
    x = cwt_tensor.unsqueeze(0).unsqueeze(0)  # (1, 1, S, T)
    x_pooled = F.avg_pool2d(x, kernel_size=(1, downsample_factor), stride=(1, downsample_factor))
    cwt_downsampled = x_pooled.squeeze(0).squeeze(0)  # (S, T//ds)
    return cwt_downsampled


# 处理单条记录：裁成 30 个窗口，每个窗口做 CWT
def process_record(record_tensor, num_windows=30, fs=128, minutes=30, num_scales=36, downsample_factor=8):
    total_length = fs * 60 * minutes  # 应为 230400
    window_size = total_length // num_windows  # 一分钟一个窗口

    assert record_tensor.shape[0] == total_length, f"record shape {record_tensor.shape}, expected {total_length}"
    
    results = []
    for i in range(num_windows):
        start = i * window_size
        end = (i + 1) * window_size
        window = record_tensor[start:end]
        cwt_result = apply_cwt(window, num_scales=num_scales, downsample_factor=downsample_factor)
        results.append(cwt_result)  # shape: (36, T//ds)
    
    return torch.stack(results)  # shape: (30, 36, T//ds)


# ====== 批量处理 DataLoader 所有记录 ======


all_processed = []
all_labels = []

for batch_inputs, batch_labels in tqdm(dataloader, desc="Processing CWT for all records"):
    batch_inputs = batch_inputs.squeeze(-1)  # shape: (B, 230400)
    batch_processed = []

    for i in range(batch_inputs.shape[0]):
        result = process_record(batch_inputs[i])  # shape: (30, 36, 960)
        batch_processed.append(result)

    batch_processed = torch.stack(batch_processed)  # shape: (B, 30, 36, 960)
    all_processed.append(batch_processed)
    all_labels.append(batch_labels)

# 拼接所有结果
X_cwt = torch.cat(all_processed, dim=0)  # shape: (1248, 30, 36, 960)
y_all = torch.cat(all_labels, dim=0)     # shape: (1248, 30)

print("最终处理后数据：", X_cwt.shape, y_all.shape)

In [ ]:
# 保存结果
torch.save(X_cwt, 'X_cwt_all.pt')
torch.save(y_all, 'y_all.pt')

In [ ]:
import torch
X_cwt = torch.load("X_cwt_all.pt")  # 或之前你保存好的数据
y_all = torch.load("y_all.pt")  # 或之前你保存好的数据
def check_nan(name, tensor):
    if torch.isnan(tensor).any():
        print(f"[!] NaN detected in {name}")
    if torch.isinf(tensor).any():
        print(f"[!] Inf detected in {name}")

check_nan("X_cwt", X_cwt)
check_nan("y_all", y_all)

In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch.nn.functional as F


# ====== STFT 函数（频率池化到 36，时间下采样到 960）======
def apply_stft(signal, n_fft=256, hop_length=64, target_freq_bins=36, target_time_bins=960):
    """
    signal: torch.Tensor, shape (T,)
    return: torch.Tensor, shape (36, 960)
    """
    # 计算 STFT
    stft_result = torch.stft(
        signal, 
        n_fft=n_fft, 
        hop_length=hop_length, 
        win_length=n_fft,
        return_complex=True
    )  # (F, T_frames)

    magnitude = torch.abs(stft_result)  # (F, T_frames)

    # 增加 batch/channel 维度 -> (1,1,F,T)
    x = magnitude.unsqueeze(0).unsqueeze(0)

    # 频率池化到 36，时间池化到 960
    x_resized = F.adaptive_avg_pool2d(x, (target_freq_bins, target_time_bins))  # (1,1,36,960)

    return x_resized.squeeze(0).squeeze(0)  # (36, 960)


# ====== 处理单条记录：裁成 30 个窗口，每个窗口做 STFT ======
def process_record_stft(record_tensor, num_windows=30, fs=128, minutes=30, 
                        n_fft=256, hop_length=64, target_freq_bins=36, target_time_bins=960):
    total_length = fs * 60 * minutes  # 230400
    window_size = total_length // num_windows

    assert record_tensor.shape[0] == total_length, f"record shape {record_tensor.shape}, expected {total_length}"
    
    results = []
    for i in range(num_windows):
        start = i * window_size
        end = (i + 1) * window_size
        window = record_tensor[start:end]
        stft_result = apply_stft(window, n_fft=n_fft, hop_length=hop_length, 
                                 target_freq_bins=target_freq_bins, target_time_bins=target_time_bins)
        results.append(stft_result)  # (36, 960)
    
    return torch.stack(results)  # (30, 36, 960)


# ====== 批量处理 DataLoader 所有记录 ======
all_processed = []
all_labels = []

for batch_inputs, batch_labels in tqdm(dataloader, desc="Processing STFT for all records"):
    batch_inputs = batch_inputs.squeeze(-1)  # (B, 230400)
    batch_processed = []

    for i in range(batch_inputs.shape[0]):
        result = process_record_stft(batch_inputs[i])  # (30, 36, 960)
        batch_processed.append(result)

    batch_processed = torch.stack(batch_processed)  # (B, 30, 36, 960)
    all_processed.append(batch_processed)
    all_labels.append(batch_labels)

# 拼接
X_stft = torch.cat(all_processed, dim=0)  # (1248, 30, 36, 960)
y_all = torch.cat(all_labels, dim=0)      # (1248, 30)

print("最终处理后数据：", X_stft.shape, y_all.shape)


In [ ]:
# 保存结果
torch.save(X_stft, '/data/chenj/X_stft_all.pt')
torch.save(y_all, '/data/chenj/y_stft_all.pt')

In [ ]:
import torch
X_stft = torch.load("/data/chenj/X_stft_all.pt")  # 或之前你保存好的数据
y_stft__all = torch.load("/data/chenj/y_stft_all.pt")  # 或之前你保存好的数据
def check_nan(name, tensor):
    if torch.isnan(tensor).any():
        print(f"[!] NaN detected in {name}")
    if torch.isinf(tensor).any():
        print(f"[!] Inf detected in {name}")

check_nan("X_stft", X_stft)
check_nan("y_stft__all", y_stft__all)

In [ ]:
count = 0
for i in range(X_stft.shape[0]):
    if torch.isnan(X_stft[i]).any():
        print(f"NaN in sample {i}")
        count += 1
print(f"Total samples with NaN: {count}")


### 1.7 3层残差TCN嵌入

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualTCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super(ResidualTCNBlock, self).__init__()
        padding = (kernel_size - 1) * dilation // 2

        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              padding=padding, dilation=dilation)
        self.norm = nn.LayerNorm(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

        self.resample = nn.Conv1d(in_channels, out_channels, kernel_size=1) \
            if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        # x shape: (B, C, T)
        residual = self.resample(x)
        out = self.conv(x)                     # (B, C_out, T)
        out = self.relu(out)
        out = self.dropout(out)

        # Permute for LayerNorm: (B, C, T) -> (B, T, C)
        out = out.permute(0, 2, 1)
        out = self.norm(out)
        out = out.permute(0, 2, 1)

        return out + residual                  # Residual connection


class ImprovedTCN(nn.Module):
    def __init__(self, input_channels=1, hidden_channels=128, output_channels=512,
                 kernel_size=3, num_layers=3, dropout=0.2):
        super(ImprovedTCN, self).__init__()
        layers = []
        for i in range(num_layers):
            in_ch = input_channels if i == 0 else hidden_channels
            out_ch = output_channels if i == num_layers - 1 else hidden_channels
            dilation = 2 ** i
            layers.append(ResidualTCNBlock(in_ch, out_ch, kernel_size, dilation, dropout))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # x: (B, 1, T)
        return self.network(x)  # (B, C, T)


class CWTEmbeddingModel(nn.Module):
    def __init__(self, input_length=960, embed_dim=512):
        super(CWTEmbeddingModel, self).__init__()
        self.tcn = ImprovedTCN(input_channels=1,
                               hidden_channels=128,
                               output_channels=embed_dim,
                               kernel_size=3,
                               num_layers=3,
                               dropout=0.2)

    def forward(self, x):
        """
        Input:  x shape: (N, 30, 36, 960)
        Output: shape: (N, 30, 36, 512)
        """
        B, W, F, T = x.shape
        x = x.view(B * W * F, 1, T)                     # (B*30*36, 1, 960)
        x_embed = self.tcn(x)                           # → (B*30*36, 512, T')
        x_embed = F.adaptive_avg_pool1d(x_embed, 1)     # → (B*30*36, 512, 1)
        x_embed = x_embed.squeeze(-1)                   # → (B*30*36, 512)
        x_embed = x_embed.view(B, W, F, -1)             # → (B, 30, 36, 512)
        return x_embed


In [ ]:
device = torch.device(f"cuda:{1}" if torch.cuda.is_available() else "cpu")
embedding_model = CWTEmbeddingModel().to(device)
embedding_model.eval()  # 不进行 dropout 等操作

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

X_cwt = torch.load("X_cwt_all.pt")  # 或之前你保存好的数据
batch_size = 1
loader = DataLoader(X_cwt, batch_size=batch_size, shuffle=False)

all_embeddings = []

with torch.no_grad():
    for batch in tqdm(loader):
        # batch shape: (B, 30, 36, 960)
        B, W, F, T = batch.shape
        batch = batch.view(B * W * F, 1, T).to(device)  # -> (B*30*36, 1, 960)

        out = embedding_model.tcn(batch)           # (B*W*F, 512, T')
        out = torch.nn.functional.adaptive_avg_pool1d(out, 1).squeeze(-1)  # (B*W*F, 512)
        out = out.view(B, W, F, -1)  # -> (B, 30, 36, 512)

        all_embeddings.append(out.cpu())  # 可用 .cuda() 提升速度

# 拼接所有结果
X_embedded = torch.cat(all_embeddings, dim=0)  # -> (1248, 30, 36, 512)
print(X_embedded.shape)

# 保存结果
torch.save(X_embedded, 'X_cwt_embedded.pt')


# 2. 实验

In [ ]:
import torch
X_embedded = torch.load("/data/chenj/X_cwt_embedded.pt")  # 或之前你保存好的数据
y_all= torch.load("y_all.pt")  # 或之前你保存好的数据

In [ ]:
X_embedded.shape  # 查看数据形状

In [ ]:
y_all.shape  # 查看标签形状

In [ ]:
def check_nan(name, tensor):
    if torch.isnan(tensor).any():
        print(f"[!] NaN detected in {name}")
    if torch.isinf(tensor).any():
        print(f"[!] Inf detected in {name}")

check_nan("X_embedded", X_embedded)
check_nan("y_all", y_all)

In [ ]:
# 假设数据变量为 X_embedded，shape: (1248, 30, 36, 512)

# 低频段
X_low = X_embedded[:, :, 0:12, :]     # shape: (1248, 30, 12, 512)

# 中频段
X_mid = X_embedded[:, :, 12:24, :]    # shape: (1248, 30, 12, 512)

# 高频段
X_high = X_embedded[:, :, 24:36, :]   # shape: (1248, 30, 12, 512)

print(type(X_low))  # 应该是 <class 'torch.Tensor'>

In [ ]:
#  X_low 是 [1248, 30, 12, 512]
X_low_pooled  = X_low.mean(dim=2)   # => [1248, 30, 512]
X_mid_pooled  = X_mid.mean(dim=2)   # => [1248, 30, 512]
X_high_pooled = X_high.mean(dim=2)  # => [1248, 30, 512]

print(X_low_pooled.shape)  # 应该是 torch.Size([1248, 30, 512])

### Top k 相似度图（动态图）

In [ ]:
import torch
import torch.nn.functional as F

def build_topk_graph(node_features, topk=5, eps=1e-6):
    """
    构建无向图的邻接矩阵（cosine相似度 + TopK + 对称化）

    Args:
        node_features: Tensor，形状为 [B, N, F]，B是batch，N是节点数，F是特征维度
        topk: 每个节点保留的TopK相似节点
        eps: 防止除以0

    Returns:
        adj: 无向图邻接矩阵，形状 [B, N, N]
    """

    B, N, _ = node_features.shape

    # 归一化
    x_norm = F.normalize(node_features, p=2, dim=-1)  # [B, N, F]

    # 计算余弦相似度（批量矩阵乘法）
    sim_matrix = torch.matmul(x_norm, x_norm.transpose(1, 2))  # [B, N, N]

    # 保留TopK（不包括自身）
    topk_values, topk_indices = torch.topk(sim_matrix, k=topk+1, dim=-1)  # +1是因为包括自己

    # 创建一个零矩阵作为掩码
    mask = torch.zeros_like(sim_matrix)

    # 用scatter填充TopK的值（batch-wise）
    batch_indices = torch.arange(B).view(B, 1, 1).expand(B, N, topk+1)
    node_indices  = torch.arange(N).view(1, N, 1).expand(B, N, topk+1)

    mask[batch_indices, node_indices, topk_indices] = 1.0

    # 剪掉自连接（可选）
    mask = mask * (1 - torch.eye(N, device=mask.device).unsqueeze(0))

    # 只保留topk对应的相似度值
    adj = sim_matrix * mask  # [B, N, N]

    # 转为无向图：对称化
    adj = 0.5 * (adj + adj.transpose(1, 2))

    return adj


In [ ]:
'''
# ---------- ✅ 测试 ----------
if __name__ == "__main__":
    B = 4   # batch size
    N = 30  # number of nodes (时间窗口)
    F_dim = 512  # 特征维度

    # 随机输入特征：形状 [B, N, F]
    node_features = torch.randn(B, N, F_dim)

    # 构建TopK图
    adj = build_topk_graph(node_features, topk=5)

    # 打印信息
    print("输入 shape:", node_features.shape)
    print("输出邻接矩阵 shape:", adj.shape)  # 应该是 [B, N, N]
    print("邻接矩阵中非零元素数量（稀疏度）:")
    for b in range(B):
        nonzero = torch.count_nonzero(adj[b])
        print(f" - Batch {b}: {nonzero.item()} / {N*N}")

    # 可选：查看一个样本邻接矩阵的可视化（使用matplotlib）
    try:
        import matplotlib.pyplot as plt
        plt.imshow(adj[0].cpu().numpy(), cmap='hot')
        plt.title("Sample Adjacency Matrix (Batch 0)")
        plt.colorbar()
        plt.show()
    except ImportError:
        pass
'''

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchdiffeq import odeint

def normalize_adj(adj):
    """
    对批量邻接矩阵做对称归一化：A_hat = D^(-1/2) (A+I) D^(-1/2)
    adj: Tensor, shape (B, N, N)
    return: Tensor, shape (B, N, N)
    """
    B, N, _ = adj.shape
    device = adj.device

    I = torch.eye(N, device=device).unsqueeze(0).expand(B, -1, -1)
    A_hat = adj + I  # 加自环

    D = A_hat.sum(dim=2)  # 度矩阵 (B, N)
    D_inv_sqrt = torch.pow(D, -0.5)
    D_inv_sqrt[torch.isinf(D_inv_sqrt)] = 0.0

    D_inv_sqrt = torch.diag_embed(D_inv_sqrt)  # (B, N, N)

    return torch.bmm(torch.bmm(D_inv_sqrt, A_hat), D_inv_sqrt)  # D^-0.5 * A_hat * D^-0.5

class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, adj):
        """
        x: Tensor, (B, N, F_in)
        adj: Tensor, (B, N, N) 未归一化邻接矩阵
        """
        adj_norm = normalize_adj(adj)  # 规范化邻接矩阵
        support = self.linear(x)       # 线性变换 (B, N, F_out)
        out = torch.bmm(adj_norm, support)  # 邻居特征聚合
        return F.relu(out)


In [ ]:
from torchdiffeq import odeint
class ODEFunc(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.gcn1 = GCNLayer(hidden_dim, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, hidden_dim)

    def forward(self, t, x):
        # t 是时间参数，x 是 (B, N, F)
        out = self.gcn1(x, self.adj)
        out = self.gcn2(out, self.adj)
        return out

    def set_adj(self, adj):
        self.adj = adj


class ODEBlock(nn.Module):
    def __init__(self, odefunc, tol=1e-3):
        super().__init__()
        self.odefunc = odefunc
        self.tol = tol
        self.register_buffer('integration_time', torch.tensor([0., 1.], dtype=torch.float32))

    def forward(self, x, adj):
        self.odefunc.set_adj(adj)

        x = x.to(dtype=torch.float32)
        t = self.integration_time.to(x.device)

        # debug
        if (t[1] - t[0]).item() <= 0:
            print(f"[!] ERROR: dt = 0: t = {t}")

        out = odeint(
            self.odefunc,
            x,
            t,
            rtol=self.tol,
            atol=self.tol,
            method='dopri5'
        )
        return out[1]


class ODEGCNModel(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.odefunc = ODEFunc(hidden_dim)
        self.odeblock = ODEBlock(self.odefunc)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, adj):
        # x: (B, N, input_dim)
        x = self.input_proj(x)               # (B, N, hidden_dim)
        x = self.odeblock(x, adj)            # (B, N, hidden_dim)
        logits = self.classifier(x).squeeze(-1)  # (B, N)
        return torch.sigmoid(logits)

In [ ]:
import torch
import torch.nn as nn

class MultiBandODEGCN(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128):
        super().__init__()
        # 3个频带分别建模，复用同一个ODEGCNModel结构（你也可以改成不同的）
        self.model_low = ODEGCNModel(input_dim, hidden_dim)
        self.model_mid = ODEGCNModel(input_dim, hidden_dim)
        self.model_high = ODEGCNModel(input_dim, hidden_dim)

        # 融合后做分类
        self.fusion_linear = nn.Linear(hidden_dim * 3, 1)

    def forward(self, x_low, adj_low, x_mid, adj_mid, x_high, adj_high):
        """
        x_low, x_mid, x_high: (B, N, input_dim)
        adj_low, adj_mid, adj_high: (B, N, N)
        """
        feat_low = self.model_low.input_proj(x_low)
        feat_low = self.model_low.odeblock(feat_low, adj_low)  # (B, N, hidden_dim)

        feat_mid = self.model_mid.input_proj(x_mid)
        feat_mid = self.model_mid.odeblock(feat_mid, adj_mid)  # (B, N, hidden_dim)

        feat_high = self.model_high.input_proj(x_high)
        feat_high = self.model_high.odeblock(feat_high, adj_high)  # (B, N, hidden_dim)

        # concat融合
        feat_concat = torch.cat([feat_low, feat_mid, feat_high], dim=-1)  # (B, N, hidden_dim*3)

        # 全连接降维融合
        logits = self.fusion_linear(feat_concat).squeeze(-1)  # (B, N)
        preds = torch.sigmoid(logits)
        return preds


In [ ]:
device = torch.device(f"cuda:{1}" if torch.cuda.is_available() else "cpu")
# 构建邻接矩阵
adj_low = build_topk_graph(X_low_pooled, topk=3)

adj_mid = build_topk_graph(X_mid_pooled, topk=3)

adj_high = build_topk_graph(X_high_pooled, topk=3)

model = MultiBandODEGCN(input_dim=512, hidden_dim=128)
model = model.to(device)  # 再次 to(device) 保证所有模块移动
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MultiBandECGDataset(Dataset):
    def __init__(self, X_low, X_mid, X_high, y):
        self.X_low = X_low
        self.X_mid = X_mid
        self.X_high = X_high
        self.y = y

    def __len__(self):
        return self.X_low.shape[0]

    def __getitem__(self, idx):
        return (self.X_low[idx], self.X_mid[idx], self.X_high[idx]), self.y[idx]

# 假设你的 X 和 y 都是 tensor 并已加载到 GPU 或 CPU
batch_size = 32
dataset = MultiBandECGDataset(X_low_pooled, X_mid_pooled, X_high_pooled, y_all)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [ ]:
def check_tensor(tensor, name="tensor"):
    if torch.isnan(tensor).any():
        print(f"[!] NaN detected in {name}")
    if torch.isinf(tensor).any():
        print(f"[!] Inf detected in {name}")
    print(f"{name}.shape: {tensor.shape}, device: {tensor.device}, dtype: {tensor.dtype}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, f1_score, mean_squared_error, confusion_matrix

def evaluate_metrics(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred > threshold).astype(int)
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred_bin.flatten()

    acc = accuracy_score(y_true_flat, y_pred_flat)
    recall = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
    return acc, recall, f1



def evaluate_metrics_yelongzhu(y_true, y_pred_prob, threshold=0.5):
    # 将预测概率阈值化为标签
    y_pred = (y_pred_prob >= threshold).astype(int)

    acc = accuracy_score(y_true.flatten(), y_pred.flatten())
    f1 = f1_score(y_true.flatten(), y_pred.flatten())
    mse = mean_squared_error(y_true.flatten(), y_pred_prob.flatten())  # 注意还是用概率来算

    # 计算混淆矩阵
    tn, fp, fn, tp = confusion_matrix(y_true.flatten(), y_pred.flatten()).ravel()

    fpr = fp / (fp + tn + 1e-8)
    fnr = fn / (fn + tp + 1e-8)

    return acc, f1, mse, fpr, fnr


def evaluate_minute_accuracy(y_true, y_pred_prob, threshold=0.5, save_path=None):
    """
    时间维度评估函数：返回全局指标 + 逐分钟准确率
    
    参数：
        y_true: 真实标签数组 (n_samples, 30)
        y_pred_prob: 预测概率数组 (n_samples, 30)
        threshold: 分类阈值
        
    返回：
        global_acc: 全局准确率
        minute_acc_df: 逐分钟准确率DataFrame
    """
    # 验证输入维度
    assert y_true.shape == y_pred_prob.shape, "输入维度不一致"
    assert y_true.shape[1] == 30, "时间维度应为30分钟"
    
    # 生成预测标签
    y_pred = (y_pred_prob >= threshold).astype(int)

     # 计算全局准确率（所有样本所有时间点）
    global_acc = accuracy_score(y_true.flatten(), y_pred.flatten())
    
    # 逐分钟计算准确率
    minute_acc = []
    for minute in range(30):
        acc = accuracy_score(y_true[:, minute], y_pred[:, minute])
        minute_acc.append(acc)
    
    # 构建结果DataFrame
    acc_df = pd.DataFrame({
        'minute': range(1, 31),
        'accuracy': minute_acc,
        'pred_positive_rate': y_pred.mean(axis=0),  # 预测阳性率
        'true_positive_rate': y_true.mean(axis=0)   # 真实阳性率
    })

    if save_path:
        # 保存全局指标
        with open(save_path + '_global.txt', 'w') as f:
            f.write(f"全局准确率：{global_acc:.4f}\n")
            f.write(f"评估样本数：{y_true.shape[0]}\n")
            f.write(f"评估时间：{pd.Timestamp.now()}\n")

        # 保存详细数据
        acc_df.to_csv(save_path + '_detail.csv', index=False)
        acc_df.to_excel(save_path + '_detail.xlsx', index=False)

    return global_acc, acc_df



In [ ]:
import os
epochs = 800
best_f1 = 0.0
last_epoch_model_path = None  # 保存上一个保存的模型路径

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_targets = []

    for (x_low_batch, x_mid_batch, x_high_batch), y_batch in dataloader:
        x_low_batch = x_low_batch.to(device)   # (B, 30, 12, 512)
        x_mid_batch = x_mid_batch.to(device)
        x_high_batch = x_high_batch.to(device)
        y_batch = y_batch.to(device)           # (B, 30)

        # 构建邻接矩阵并对称化
        adj_low = build_topk_graph(x_low_batch, topk=5).to(device)
        adj_low = 0.5 * (adj_low + adj_low.transpose(1, 2))

        adj_mid = build_topk_graph(x_mid_batch, topk=5).to(device)
        adj_mid = 0.5 * (adj_mid + adj_mid.transpose(1, 2))

        adj_high = build_topk_graph(x_high_batch, topk=5).to(device)
        adj_high = 0.5 * (adj_high + adj_high.transpose(1, 2))

        # 通过模型预测
        preds = model(x_low_batch, adj_low, x_mid_batch, adj_mid, x_high_batch, adj_high)  # (B, 30)

        # 计算 loss
        loss = loss_fn(preds, y_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss = total_loss / batch_size

        # 阈值化为 0/1 的整数标签
        preds_bin = (preds > 0.5).int()  # shape: (B, 30), int
        # print(preds_bin[0])

        # 收集预测与真实标签
        all_preds.append(preds_bin.detach().cpu().numpy())
        all_targets.append(y_batch.int().cpu().numpy())  # 保证和预测格式一致

    # 合并所有batch的预测和标签
    y_pred_all = np.concatenate(all_preds, axis=0)  # shape: (N, 30)
    y_true_all = np.concatenate(all_targets, axis=0)

    acc, f1, mse, fpr, fnr = evaluate_metrics_yelongzhu(y_true_all.flatten(), y_pred_all.flatten())
    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | MSE: {mse:.4f} | FPR: {fpr:.4f} | FNR: {fnr:.4f}")
    
    # 调用函数并保存结果到当前目录
    global_acc, minute_df = evaluate_minute_accuracy(
        y_true_all,
        y_pred_all,
        save_path='./minute_accuracy_report'  # 自动添加后缀
    )

     # ✅ 每20个epoch保存模型，并删除上一个
    if (epoch + 1) % 20 == 0:
        model_path = f"model_epoch_{epoch+1}.pt"
        torch.save(model.state_dict(), model_path)
        print(f"✅ 模型已保存：{model_path}")

        # 删除上一个保存的模型
        if last_epoch_model_path and os.path.exists(last_epoch_model_path):
            os.remove(last_epoch_model_path)
            print(f"🗑️ 删除旧模型：{last_epoch_model_path}")

        last_epoch_model_path = model_path

    # ✅ 保存F1最佳模型
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pt")
        print(f"🏆 最佳模型更新 (F1={f1:.4f})，保存为 best_model.pt")
    

In [ ]:
print("预测结果形状:", y_pred_all.shape)  # 应该是 (1248, 30)
print("真实标签形状:", y_true_all.shape)  # 应该是 (1248, 30)
print("y_pred_all:", y_pred_all[:5])  # 打印前5个预测结果
print("y_true_all:", y_true_all[:5])  # 打印前5个

In [ ]:
# 评估原始预测 
# 第一次出现后置为1，出现前置为0（存在偶然事件）

'''
# 后处理：一旦出现 1，后面全部置为 1
y_pred_all_post = y_pred_all.copy()

for i in range(y_pred_all_post.shape[0]):  # 遍历每个样本
    ones_idx = np.where(y_pred_all_post[i] == 1)[0]  # 找到所有预测为 1 的位置
    if len(ones_idx) > 0:
        first_one = ones_idx[0]  # 第一次出现 1 的位置
        y_pred_all_post[i, first_one:] = 1  # 之后全设为 1
'''

In [ ]:
# 评估原始预测 
# 连续3次出现1后全置为1，出现前置为0；
# 如果没有连续 3 个 1 → 再找连续 2 个 1，同样处理
# 如果没有2次，找第一次出现1后全置为1；
# 如果没有1，全部置为0；

import numpy as np

def postprocess_predictions(y_pred_all, m=3):
    y_pred_all_post = y_pred_all.copy()

    for i in range(y_pred_all_post.shape[0]):  # 遍历每个样本
        pred = y_pred_all_post[i]
        trigger_idx = -1

        # Step 1: 找连续 m=3 个 1
        for j in range(len(pred) - m + 1):
            if np.all(pred[j:j+m] == 1):
                trigger_idx = j
                break

        # Step 2: 如果没有，找连续 2 个 1
        if trigger_idx == -1:
            for j in range(len(pred) - 1):
                if np.all(pred[j:j+2] == 1):
                    trigger_idx = j
                    break

        # Step 3: 如果还是没有，找第一个 1
        if trigger_idx == -1:
            ones_idx = np.where(pred == 1)[0]
            if len(ones_idx) > 0:
                trigger_idx = ones_idx[0]

        # Step 4: 修改结果
        if trigger_idx != -1:
            pred[:trigger_idx] = 0
            pred[trigger_idx:] = 1
        else:
            pred[:] = 0  # 没有任何 1

        y_pred_all_post[i] = pred

    return y_pred_all_post

y_pred_all_modified = postprocess_predictions(y_pred_all, m=3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 假设你已经有 y_pred_all 和 y_true_all
# 这里选择前 5 个样本来画图（避免一次性太多）
num_samples_to_plot = 5
time_steps = np.arange(1, 31)  # x轴：1 到 30 分钟

plt.figure(figsize=(12, 8))

for i in range(num_samples_to_plot):
    plt.subplot(num_samples_to_plot, 1, i + 1)
    plt.plot(time_steps, y_pred_all_modified[-i], color='blue', label='predicted')
    plt.plot(time_steps, y_true_all[-i], color='red',  label='true')
    plt.ylim(-0.1, 1.1)  # 让 0/1 看起来更清晰
    plt.yticks([0, 1])
    plt.title(f'Sample {i+1}')
    if i == num_samples_to_plot - 1:
        plt.xlabel('minute')
    plt.ylabel('SCD ?')

    if i == 0:  # 只在第一个子图画图例
        plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

# 假设 y_pred_all 和 y_true_all 都是 (N, 30) 0/1 数组

early_count = 0
late_count = 0
same_count = 0
no_event_count = 0  # 真实标签中没有 1 的样本数

early_deltas = []
late_deltas = []

for i in range(y_pred_all_modified.shape[0]):
    # 找第一次出现 1 的位置
    pred_idx = np.where(y_pred_all_modified[i] == 1)[0]
    true_idx = np.where(y_true_all[i] == 1)[0]

    if len(true_idx) == 0:
        # 如果真实数据中没有 1，则跳过
        no_event_count += 1
        continue

    true_first = true_idx[0]
    pred_first = pred_idx[0] if len(pred_idx) > 0 else None

    if pred_first is None:
        # 预测中没有 1，视为晚到末尾
        late_count += 1
        late_deltas.append(30 - true_first)
    else:
        diff = pred_first - true_first
        if diff < 0:
            early_count += 1
            early_deltas.append(-diff)  # 提前多少分钟
        elif diff > 0:
            late_count += 1
            late_deltas.append(diff)  # 晚多少分钟
        else:
            same_count += 1  # 刚好同一分钟

# 打印统计结果
print("===== 预测与真实首次出现1的时间对比 =====")
print(f"提前出现的次数: {early_count}，平均提前 {np.mean(early_deltas):.2f} 分钟" if early_count > 0 else "提前出现的次数: 0")
print(f"延迟出现的次数: {late_count}，平均延迟 {np.mean(late_deltas):.2f} 分钟" if late_count > 0 else "延迟出现的次数: 0")
print(f"与真实同时出现的次数: {same_count}")
print(f"真实无事件（全为0）的样本数: {no_event_count}")


In [ ]:
import os
epochs = 200
best_f1 = 0.75
last_epoch_model_path = 'model_epoch_800.pt'  # 保存上一个保存的模型路径

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_targets = []

    for (x_low_batch, x_mid_batch, x_high_batch), y_batch in dataloader:
        x_low_batch = x_low_batch.to(device)   # (B, 30, 12, 512)
        x_mid_batch = x_mid_batch.to(device)
        x_high_batch = x_high_batch.to(device)
        y_batch = y_batch.to(device)           # (B, 30)

        # 构建邻接矩阵并对称化
        adj_low = build_topk_graph(x_low_batch, topk=5).to(device)
        adj_low = 0.5 * (adj_low + adj_low.transpose(1, 2))

        adj_mid = build_topk_graph(x_mid_batch, topk=5).to(device)
        adj_mid = 0.5 * (adj_mid + adj_mid.transpose(1, 2))

        adj_high = build_topk_graph(x_high_batch, topk=5).to(device)
        adj_high = 0.5 * (adj_high + adj_high.transpose(1, 2))

        # 通过模型预测
        preds = model(x_low_batch, adj_low, x_mid_batch, adj_mid, x_high_batch, adj_high)  # (B, 30)

        # 计算 loss
        loss = loss_fn(preds, y_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss = total_loss / batch_size

        # 阈值化为 0/1 的整数标签
        preds_bin = (preds > 0.5).int()  # shape: (B, 30), int
        # print(preds_bin[0])

        # 收集预测与真实标签
        all_preds.append(preds_bin.detach().cpu().numpy())
        all_targets.append(y_batch.int().cpu().numpy())  # 保证和预测格式一致

    # 合并所有batch的预测和标签
    y_pred_all = np.concatenate(all_preds, axis=0)  # shape: (N, 30)
    y_true_all = np.concatenate(all_targets, axis=0)

    y_pred_all_modified = postprocess_predictions(y_pred_all, m=3)

    acc, f1, mse, fpr, fnr = evaluate_metrics_yelongzhu(y_true_all.flatten(), y_pred_all_modified.flatten())
    print(f"Epoch {epoch+201}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | MSE: {mse:.4f} | FPR: {fpr:.4f} | FNR: {fnr:.4f}")
    
    # 调用函数并保存结果到当前目录
    global_acc, minute_df = evaluate_minute_accuracy(
        y_true_all,
        y_pred_all,
        save_path='./minute_accuracy_report'  # 自动添加后缀
    )

     # ✅ 每20个epoch保存模型，并删除上一个
    if (epoch + 801) % 20 == 0:
        model_path = f"model_epoch_{epoch+801}.pt"
        torch.save(model.state_dict(), model_path)
        print(f"✅ 模型已保存：{model_path}")

        # 删除上一个保存的模型
        if last_epoch_model_path and os.path.exists(last_epoch_model_path):
            os.remove(last_epoch_model_path)
            print(f"🗑️ 删除旧模型：{last_epoch_model_path}")

        last_epoch_model_path = model_path

    # ✅ 保存F1最佳模型
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pt")
        print(f"🏆 最佳模型更新 (F1={f1:.4f})，保存为 best_model.pt")
    

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

def visualize_acc_barchart(file_path):
    """
    分钟级准确率柱状图可视化
    :param file_path: CSV文件路径
    """
    # 数据加载与验证
    df = pd.read_csv(file_path)
    if not {'minute', 'accuracy'}.issubset(df.columns):
        raise ValueError("CS文件必须包含minute和accuracy列")

    # 创建可视化画布
    plt.figure(figsize=(18, 10))
    ax = plt.gca()
    
    # 绘制柱状图
    bars = ax.bar(
        x=df['minute'],
        height=df['accuracy'],
        width=0.6,
        color='#2c7bb6',
        alpha=0.8,
        edgecolor='white'
    )
    '''

    # 添加数据标签
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height-0.02,
                f'{height:.2%}',
                ha='center', va='top',
                color='white',
                fontsize=10,
                fontweight='bold')
    '''
    # 坐标轴设置
    ax.set_xlabel('分钟', fontsize=14, labelpad=12)
    ax.set_ylabel('准确率', fontsize=14)
    ax.set_xticks(range(1, 31))
    ax.set_xticklabels([f'{m:02d}' for m in range(1, 31)])
    ax.set_ylim(df['accuracy'].min()-0.02, df['accuracy'].max()+0.02)
    
    # 图表装饰
    plt.title('分钟级分类准确率分布\n', fontsize=16, pad=20)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# 使用示例
if __name__ == "__main__":
    # 测试路径（替换为实际路径）
    test_path = "./minute_accuracy_report_detail.csv"
    visualize_acc_barchart(test_path)

In [ ]:
epochs = 200


for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_targets = []

    for (x_low_batch, x_mid_batch, x_high_batch), y_batch in dataloader:
        x_low_batch = x_low_batch.to(device)   # (B, 30, 12, 512)
        x_mid_batch = x_mid_batch.to(device)
        x_high_batch = x_high_batch.to(device)
        y_batch = y_batch.to(device)           # (B, 30)

        # 构建邻接矩阵并对称化
        adj_low = build_topk_graph(x_low_batch, topk=5).to(device)
        adj_low = 0.5 * (adj_low + adj_low.transpose(1, 2))

        adj_mid = build_topk_graph(x_mid_batch, topk=5).to(device)
        adj_mid = 0.5 * (adj_mid + adj_mid.transpose(1, 2))

        adj_high = build_topk_graph(x_high_batch, topk=5).to(device)
        adj_high = 0.5 * (adj_high + adj_high.transpose(1, 2))

        # 通过模型预测
        preds = model(x_low_batch, adj_low, x_mid_batch, adj_mid, x_high_batch, adj_high)  # (B, 30)

        # 计算 loss
        loss = loss_fn(preds, y_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss = total_loss / batch_size

        # 阈值化为 0/1 的整数标签
        preds_bin = (preds > 0.5).int()  # shape: (B, 30), int
        # print(preds_bin[0])

        # 收集预测与真实标签
        all_preds.append(preds_bin.detach().cpu().numpy())
        all_targets.append(y_batch.int().cpu().numpy())  # 保证和预测格式一致

    # 合并所有batch的预测和标签
    y_pred_all = np.concatenate(all_preds, axis=0)  # shape: (N, 30)
    y_true_all = np.concatenate(all_targets, axis=0)

    # 评估指标（flatten 后按元素比较）
    acc, recall, f1 = evaluate_metrics(y_true_all.flatten(), y_pred_all.flatten())
    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


    

In [ ]:
epochs = 300
for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_targets = []

    for (x_low_batch, x_mid_batch, x_high_batch), y_batch in dataloader:
        x_low_batch = x_low_batch.to(device)   # (B, 30, 12, 512)
        x_mid_batch = x_mid_batch.to(device)
        x_high_batch = x_high_batch.to(device)
        y_batch = y_batch.to(device)           # (B, 30)

        # 构建邻接矩阵并对称化
        adj_low = build_topk_graph(x_low_batch, topk=5).to(device)
        adj_low = 0.5 * (adj_low + adj_low.transpose(1, 2))

        adj_mid = build_topk_graph(x_mid_batch, topk=5).to(device)
        adj_mid = 0.5 * (adj_mid + adj_mid.transpose(1, 2))

        adj_high = build_topk_graph(x_high_batch, topk=5).to(device)
        adj_high = 0.5 * (adj_high + adj_high.transpose(1, 2))

        # 通过模型预测
        preds = model(x_low_batch, adj_low, x_mid_batch, adj_mid, x_high_batch, adj_high)  # (B, 30)

        # 计算 loss
        loss = loss_fn(preds, y_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss = total_loss / batch_size

        # 阈值化为 0/1 的整数标签
        preds_bin = (preds > 0.5).int()  # shape: (B, 30), int
        # print(preds_bin[0])

        # 收集预测与真实标签
        all_preds.append(preds_bin.detach().cpu().numpy())
        all_targets.append(y_batch.int().cpu().numpy())  # 保证和预测格式一致

    # 合并所有batch的预测和标签
    y_pred_all = np.concatenate(all_preds, axis=0)  # shape: (N, 30)
    y_true_all = np.concatenate(all_targets, axis=0)

    # 评估指标（flatten 后按元素比较）
    # acc, recall, f1 = evaluate_metrics(y_true_all.flatten(), y_pred_all.flatten())
    # print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    acc, f1, mse, fpr, fnr = evaluate_metrics_yelongzhu(y_true_all.flatten(), y_pred_all.flatten())
    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | MSE: {mse:.4f} | FPR: {fpr:.4f} | FNR: {fnr:.4f}")
    
    global_acc, minute_df = evaluate_minute_accuracy(y_true_all.flatten(), y_pred_all.flatten())
    # 调用函数并保存结果到当前目录
    global_acc, minute_df = evaluate_minute_accuracy(
        y_true_all.flatten(),
        y_pred_all.flatten(),
        save_path='./minute_accuracy_report'  # 自动添加后缀
    )

    

In [ ]:
epochs = 100
for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_targets = []

    for (x_low_batch, x_mid_batch, x_high_batch), y_batch in dataloader:
        x_low_batch = x_low_batch.to(device)   # (B, 30, 12, 512)
        x_mid_batch = x_mid_batch.to(device)
        x_high_batch = x_high_batch.to(device)
        y_batch = y_batch.to(device)           # (B, 30)

        # 构建邻接矩阵并对称化
        adj_low = build_topk_graph(x_low_batch, topk=5).to(device)
        adj_low = 0.5 * (adj_low + adj_low.transpose(1, 2))

        adj_mid = build_topk_graph(x_mid_batch, topk=5).to(device)
        adj_mid = 0.5 * (adj_mid + adj_mid.transpose(1, 2))

        adj_high = build_topk_graph(x_high_batch, topk=5).to(device)
        adj_high = 0.5 * (adj_high + adj_high.transpose(1, 2))

        # 通过模型预测
        preds = model(x_low_batch, adj_low, x_mid_batch, adj_mid, x_high_batch, adj_high)  # (B, 30)

        # 计算 loss
        loss = loss_fn(preds, y_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss = total_loss / batch_size

        # 阈值化为 0/1 的整数标签
        preds_bin = (preds > 0.5).int()  # shape: (B, 30), int
        # print(preds_bin[0])

        # 收集预测与真实标签
        all_preds.append(preds_bin.detach().cpu().numpy())
        all_targets.append(y_batch.int().cpu().numpy())  # 保证和预测格式一致

    # 合并所有batch的预测和标签
    y_pred_all = np.concatenate(all_preds, axis=0)  # shape: (N, 30)
    y_true_all = np.concatenate(all_targets, axis=0)

    # 评估指标（flatten 后按元素比较）
    # acc, recall, f1 = evaluate_metrics(y_true_all.flatten(), y_pred_all.flatten())
    # print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    acc, f1, mse, fpr, fnr = evaluate_metrics_yelongzhu(y_true_all.flatten(), y_pred_all.flatten())
    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | MSE: {mse:.4f} | FPR: {fpr:.4f} | FNR: {fnr:.4f}")

    

In [ ]:
epochs = 100
for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_targets = []

    for (x_low_batch, x_mid_batch, x_high_batch), y_batch in dataloader:
        x_low_batch = x_low_batch.to(device)   # (B, 30, 12, 512)
        x_mid_batch = x_mid_batch.to(device)
        x_high_batch = x_high_batch.to(device)
        y_batch = y_batch.to(device)           # (B, 30)

        # 构建邻接矩阵并对称化
        adj_low = build_topk_graph(x_low_batch, topk=5).to(device)
        adj_low = 0.5 * (adj_low + adj_low.transpose(1, 2))

        adj_mid = build_topk_graph(x_mid_batch, topk=5).to(device)
        adj_mid = 0.5 * (adj_mid + adj_mid.transpose(1, 2))

        adj_high = build_topk_graph(x_high_batch, topk=5).to(device)
        adj_high = 0.5 * (adj_high + adj_high.transpose(1, 2))

        # 通过模型预测
        preds = model(x_low_batch, adj_low, x_mid_batch, adj_mid, x_high_batch, adj_high)  # (B, 30)

        # 计算 loss
        loss = loss_fn(preds, y_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss = total_loss / batch_size

        # 阈值化为 0/1 的整数标签
        preds_bin = (preds > 0.5).int()  # shape: (B, 30), int
        # print(preds_bin[0])

        # 收集预测与真实标签
        all_preds.append(preds_bin.detach().cpu().numpy())
        all_targets.append(y_batch.int().cpu().numpy())  # 保证和预测格式一致

    # 合并所有batch的预测和标签
    y_pred_all = np.concatenate(all_preds, axis=0)  # shape: (N, 30)
    y_true_all = np.concatenate(all_targets, axis=0)

    # 评估指标（flatten 后按元素比较）
    # acc, recall, f1 = evaluate_metrics(y_true_all.flatten(), y_pred_all.flatten())
    # print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    acc, f1, mse, fpr, fnr = evaluate_metrics_yelongzhu(y_true_all.flatten(), y_pred_all.flatten())
    print(f"Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | MSE: {mse:.4f} | FPR: {fpr:.4f} | FNR: {fnr:.4f}")

    

### 可视化
